# Lab 5: A Reusable Training Pipeline

**Before you start: go to Runtime → Change runtime type and select GPU.**

This notebook gives you a clean, well-structured PyTorch training pipeline that you can adapt directly for your course project. Read through it from top to bottom — there is nothing to fill in. The goal is that by the end of the session you understand every part of the pipeline and have started adapting it to your own project.

The notebook is structured as follows:

1. Imports and setup
2. Loading a standard dataset (CIFAR-10)
3. Defining a model (pretrained backbone + custom head)
4. The training loop
5. Evaluation and loss curves
6. Saving and loading checkpoints
7. **Using your own data** — how to swap in a custom dataset
8. (Optional) Experiment tracking with Weights & Biases
9. Adapting this pipeline for other tasks

---
## A note on code style

This notebook is intentionally explicit — we avoid magic functions that hide what's happening. Every step is written out clearly so you can understand it, modify it, and debug it when something goes wrong. As you get more comfortable, you can make things more concise.

## 1. Imports and setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models

import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
from PIL import Image

# Check that a GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# If this prints 'cpu', go to Runtime → Change runtime type and select GPU

## 2. Loading a standard dataset

We use CIFAR-10 as a concrete example — 60,000 colour images across 10 classes. 
The key steps are the same for any dataset:
1. Define transforms (preprocessing + augmentation)
2. Create dataset objects
3. Wrap them in DataLoaders

### 2.1 Transforms

Transforms define how images are preprocessed before being fed to the model.
We use different transforms for training (with augmentation) and validation (without).

**Why augment only during training?** Augmentation artificially increases dataset diversity and reduces overfitting. At validation time we want a deterministic, consistent result — no randomness.

In [ ]:
# ImageNet mean and std — use these when loading a pretrained model,
# because the model was trained with images normalised this way.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Training transforms: resize, random augmentation, then normalise
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),           # resize to match pretrained model input
    transforms.RandomHorizontalFlip(),       # randomly flip left-right
    transforms.RandomCrop(224, padding=8),   # randomly crop with padding
    transforms.ColorJitter(                  # randomly vary brightness/contrast
        brightness=0.2, contrast=0.2),
    transforms.ToTensor(),                   # convert PIL image to tensor [0, 1]
    transforms.Normalize(                    # normalise to ImageNet statistics
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD),
])

# Validation transforms: just resize and normalise — no randomness
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

### 2.2 Dataset objects and DataLoaders

In [ ]:
# Download and create CIFAR-10 dataset objects
train_dataset = datasets.CIFAR10(
    root='./data',        # where to save the downloaded data
    train=True,           # training split
    download=True,
    transform=train_transform
)

val_dataset = datasets.CIFAR10(
    root='./data',
    train=False,          # test/validation split
    download=True,
    transform=val_transform
)

# Class names for CIFAR-10
class_names = train_dataset.classes
num_classes = len(class_names)
print(f'Classes: {class_names}')
print(f'Training examples: {len(train_dataset)}')
print(f'Validation examples: {len(val_dataset)}')

In [ ]:
# DataLoaders: wrap the dataset and handle batching, shuffling, and parallel loading
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,          # shuffle training data every epoch
    num_workers=2,         # parallel data loading (use 0 if you get errors)
    pin_memory=True        # faster GPU transfer
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,         # no need to shuffle validation data
    num_workers=2,
    pin_memory=True
)

print(f'Training batches per epoch: {len(train_loader)}')
print(f'Validation batches per epoch: {len(val_loader)}')

### 2.3 Visualise a batch

Always visualise your data before training. This catches preprocessing bugs early — 
wrong normalisation, swapped channels, or incorrect labels are easy to spot visually 
and very hard to diagnose from loss curves alone.

In [ ]:
def denormalise(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Reverse ImageNet normalisation for display."""
    mean = torch.tensor(mean).view(3, 1, 1)
    std  = torch.tensor(std).view(3, 1, 1)
    return torch.clamp(tensor * std + mean, 0, 1)

# Grab one batch
images, labels = next(iter(train_loader))

# Show the first 8 images
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    img = denormalise(images[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(class_names[labels[i]])
    ax.axis('off')
plt.suptitle('Sample training images (after augmentation)', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Image tensor shape: {images.shape}')   # (batch, channels, height, width)
print(f'Label tensor shape: {labels.shape}')   # (batch,)

## 3. Defining the model

We use a pretrained ResNet-18 as our backbone. Transfer learning gives us a strong 
starting point without needing to train from scratch.

The standard approach:
1. Load the pretrained model
2. Replace the final classification layer with one that matches our number of classes
3. Optionally freeze the backbone and only train the new head first

**Why ResNet-18?** It's small, fast to train, and well-understood. For your project 
you might use a larger model (ResNet-50, EfficientNet, ViT) — the pattern is identical.

In [ ]:
def build_model(num_classes, freeze_backbone=False):
    """
    Load a pretrained ResNet-18 and replace the final layer.
    
    Args:
        num_classes:      number of output classes
        freeze_backbone:  if True, only the final layer is trained
    """
    # Load ResNet-18 pretrained on ImageNet
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    
    if freeze_backbone:
        # Freeze all parameters — none will be updated by the optimiser
        for param in model.parameters():
            param.requires_grad = False
    
    # Replace the final fully connected layer
    # model.fc is the last layer; in_features tells us its input size
    in_features = model.fc.in_features  # 512 for ResNet-18
    model.fc = nn.Linear(in_features, num_classes)
    # Note: the new layer has requires_grad=True by default
    
    return model


# Build the model and move it to GPU
model = build_model(num_classes=num_classes, freeze_backbone=False)
model = model.to(device)

# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable parameters: {trainable:,} / {total:,}')

## 4. The training loop

The training loop is the core of any deep learning project. Every iteration:
1. Gets a batch of data
2. Runs a forward pass to get predictions
3. Computes the loss
4. Runs a backward pass to compute gradients
5. Updates the weights

We also run a **validation pass** at the end of each epoch — no gradient computation, 
just measuring how well the model generalises to unseen data.

### 4.1 Loss function and optimiser

In [ ]:
# Cross-entropy loss for multi-class classification
# Note: expects raw logits (not softmax output)
criterion = nn.CrossEntropyLoss()

# Adam optimiser — a good default for most projects
# lr=1e-3 is a reasonable starting point; adjust based on your loss curves
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Learning rate scheduler: reduce LR by 10x at epochs 10 and 20
# This often improves final accuracy without much tuning
scheduler = optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=[10, 20], gamma=0.1
)

### 4.2 Training and validation functions

We separate the training and validation logic into two functions. 
This makes the code cleaner and reduces the chance of bugs 
(e.g. forgetting to disable gradients during validation).

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """
    Run one full pass over the training data.
    Returns average loss and accuracy for the epoch.
    """
    model.train()   # set model to training mode (enables dropout, batch norm updates)
    
    total_loss     = 0.0
    total_correct  = 0
    total_samples  = 0
    
    for images, labels in loader:
        # Move data to GPU
        images = images.to(device)
        labels = labels.to(device)
        
        # 1. Clear gradients from the previous step
        optimizer.zero_grad()
        
        # 2. Forward pass: compute predictions
        logits = model(images)                 # shape: (batch, num_classes)
        
        # 3. Compute loss
        loss = criterion(logits, labels)
        
        # 4. Backward pass: compute gradients
        loss.backward()
        
        # 5. Update weights
        optimizer.step()
        
        # Track metrics
        total_loss    += loss.item() * images.size(0)
        preds          = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += images.size(0)
    
    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    return avg_loss, accuracy


def validate(model, loader, criterion, device):
    """
    Run one full pass over the validation data.
    Returns average loss and accuracy.
    """
    model.eval()    # set model to evaluation mode (disables dropout, freezes batch norm)
    
    total_loss    = 0.0
    total_correct = 0
    total_samples = 0
    
    with torch.no_grad():   # disable gradient computation — saves memory and time
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            
            logits = model(images)
            loss   = criterion(logits, labels)
            
            total_loss    += loss.item() * images.size(0)
            preds          = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += images.size(0)
    
    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    return avg_loss, accuracy

### 4.3 The main training loop

In [ ]:
NUM_EPOCHS = 25

# Store metrics for plotting later
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss':   [], 'val_acc':   []
}

best_val_acc  = 0.0
best_epoch    = 0

for epoch in range(1, NUM_EPOCHS + 1):
    
    # Train
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    
    # Validate
    val_loss, val_acc = validate(
        model, val_loader, criterion, device
    )
    
    # Step the learning rate scheduler
    scheduler.step()
    
    # Record metrics
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Save the best model so far
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch   = epoch
        torch.save(model.state_dict(), 'best_model.pth')
    
    # Print progress
    current_lr = optimizer.param_groups[0]['lr']
    print(
        f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
        f'Train loss: {train_loss:.4f}, acc: {train_acc:.3f} | '
        f'Val loss: {val_loss:.4f}, acc: {val_acc:.3f} | '
        f'LR: {current_lr:.6f}'
    )

print(f'\nBest validation accuracy: {best_val_acc:.3f} at epoch {best_epoch}')

## 5. Visualising training — loss curves

Loss curves are your most important diagnostic tool. You should look at them after every 
experiment. The shape of the curves tells you:

- **Both losses decreasing**: still learning — keep training
- **Train loss low, val loss high**: overfitting — add regularisation or more data
- **Both losses high**: underfitting — increase model capacity or train longer
- **Loss oscillating**: learning rate is too high
- **Loss not moving**: learning rate is too low, or there is a bug

We will cover debugging in detail in week 8.

In [ ]:
def plot_history(history):
    epochs = range(1, len(history['train_loss']) + 1)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss curves
    ax1.plot(epochs, history['train_loss'], label='Train loss')
    ax1.plot(epochs, history['val_loss'],   label='Val loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Loss curves')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Accuracy curves
    ax2.plot(epochs, history['train_acc'], label='Train accuracy')
    ax2.plot(epochs, history['val_acc'],   label='Val accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title('Accuracy curves')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_history(history)

## 6. Saving and loading checkpoints

Always save your model. Training can be interrupted (Colab sessions time out), and 
you want to be able to resume or evaluate without retraining from scratch.

We already saved the best model during training. Here we show how to load it back.

In [ ]:
# Save a full checkpoint (model weights + optimizer state + epoch)
# This allows you to resume training exactly where you left off
checkpoint = {
    'epoch':           NUM_EPOCHS,
    'model_state':     model.state_dict(),
    'optimizer_state': optimizer.state_dict(),
    'best_val_acc':    best_val_acc,
    'history':         history,
}
torch.save(checkpoint, 'checkpoint_final.pth')
print('Checkpoint saved.')

# --- Loading the best model for evaluation ---
model_eval = build_model(num_classes=num_classes)
model_eval.load_state_dict(torch.load('best_model.pth', map_location=device))
model_eval = model_eval.to(device)
model_eval.eval()

# Confirm it loads correctly
val_loss, val_acc = validate(model_eval, val_loader, criterion, device)
print(f'Loaded best model — val accuracy: {val_acc:.3f}')

## 7. Using your own data

In your project you will almost certainly be loading your own images rather than 
a standard torchvision dataset. This section shows you how to do that with a 
custom `Dataset` class.

### Expected folder structure

The simplest structure is one folder per class:

```
data/
  train/
    cat/
      img001.jpg
      img002.jpg
      ...
    dog/
      img001.jpg
      ...
  val/
    cat/
      ...
    dog/
      ...
```

If your data is already in this format, you can use `datasets.ImageFolder` directly 
and skip the custom class below — it handles this structure automatically.

### Option A: ImageFolder (simplest — use if your data is already organised by class)

In [ ]:
# Uncomment and adapt this if your data is in the folder-per-class format above

# train_dataset = datasets.ImageFolder(
#     root='data/train',
#     transform=train_transform
# )
# val_dataset = datasets.ImageFolder(
#     root='data/val',
#     transform=val_transform
# )

# class_names = train_dataset.classes
# print(f'Found classes: {class_names}')

### Option B: Custom Dataset class (use when your data has a different structure)

If your images and labels are stored differently — for example, a CSV file with 
image paths and labels, or images named with their class — you need to write a 
custom `Dataset` class. The pattern is always the same: implement `__len__` and `__getitem__`.

In [ ]:
class CustomImageDataset(Dataset):
    """
    A general-purpose image dataset that loads images from a list of file paths.
    
    Args:
        image_paths:  list of paths to image files
        labels:       list of integer class labels (same length as image_paths)
        transform:    torchvision transforms to apply
    """
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels      = labels
        self.transform   = transform
        
        assert len(image_paths) == len(labels), \
            'image_paths and labels must have the same length'
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # Load image
        image = Image.open(self.image_paths[idx]).convert('RGB')
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        # Return image and label
        label = self.labels[idx]
        return image, label


# Example usage — adapt paths and labels to your own data:
#
# image_paths = ['data/cat/001.jpg', 'data/dog/001.jpg', ...]
# labels      = [0, 1, ...]   # 0 = cat, 1 = dog
#
# train_dataset = CustomImageDataset(
#     image_paths=train_paths,
#     labels=train_labels,
#     transform=train_transform
# )

print('CustomImageDataset class defined — adapt the example usage above for your project.')

### Sanity check your custom dataset

Whenever you create a new dataset, run these checks before training. They catch the 
most common data bugs immediately.

In [ ]:
def sanity_check_dataset(dataset, class_names, num_samples=8):
    """
    Basic sanity checks for any dataset.
    Run this before training whenever you load new data.
    """
    print(f'Dataset size: {len(dataset)}')
    
    # Check a single item
    image, label = dataset[0]
    print(f'Image tensor shape: {image.shape}')   # should be (3, H, W)
    print(f'Image value range:  [{image.min():.2f}, {image.max():.2f}]')
    print(f'Label example:      {label} ({class_names[label]})')
    
    # Check class balance
    from collections import Counter
    all_labels = [dataset[i][1] for i in range(len(dataset))]
    counts = Counter(all_labels)
    print('\nClass distribution:')
    for idx, name in enumerate(class_names):
        print(f'  {name}: {counts[idx]} images')
    
    # Visualise a few samples
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for i, ax in enumerate(axes.flat):
        if i >= len(dataset):
            break
        img, lbl = dataset[i]
        img = denormalise(img).permute(1, 2, 0).numpy()
        ax.imshow(img)
        ax.set_title(class_names[lbl])
        ax.axis('off')
    plt.suptitle('Dataset sanity check — do labels look correct?', fontsize=12)
    plt.tight_layout()
    plt.show()

# Run the sanity check on our CIFAR-10 dataset
sanity_check_dataset(train_dataset, class_names)

## 8. (Optional) Experiment tracking with Weights & Biases

As your experiments get more complex, keeping track of what you tried and what worked 
becomes difficult with just print statements. Weights & Biases (W&B) is a free tool 
that logs your metrics, hyperparameters, and model outputs to a dashboard you can 
access from anywhere.

This section is optional — the pipeline above works perfectly without it. But if you 
find yourself running many experiments and losing track of results, W&B is worth 
the 10-minute setup.

In [ ]:
# To use W&B:
# 1. Run: !pip install wandb
# 2. Create a free account at wandb.ai
# 3. Uncomment and run the code below

# import wandb

# wandb.init(
#     project='my-project-name',
#     config={
#         'learning_rate': 1e-3,
#         'batch_size':    BATCH_SIZE,
#         'num_epochs':    NUM_EPOCHS,
#         'model':         'resnet18',
#     }
# )

# Then inside your training loop, replace the print statement with:
# wandb.log({
#     'epoch':      epoch,
#     'train_loss': train_loss,
#     'train_acc':  train_acc,
#     'val_loss':   val_loss,
#     'val_acc':    val_acc,
# })

print('W&B integration is optional — see comments above.')

## 9. Adapting this pipeline for other tasks

This pipeline is built around image classification, which covers the majority of 
projects. If your project involves a different task, here is what needs to change:

---

### Object detection
The model output changes from a single class label per image to a set of bounding 
boxes and class labels. The loss function changes accordingly (classification loss + 
bounding box regression loss). The DataLoader needs to return bounding box annotations 
alongside images. A good starting point is `torchvision.models.detection` which 
provides pretrained Faster R-CNN and YOLO-style models with a standard interface.

**What stays the same:** data loading pattern, training loop structure, checkpointing, 
loss curve plotting.

---

### Semantic segmentation
The model output changes from a single class label per image to a class label per 
pixel — a tensor of shape `(batch, num_classes, H, W)`. The loss function is still 
cross-entropy but applied pixel-wise. Labels must be segmentation masks rather than 
single integers. `torchvision.models.segmentation` provides pretrained DeepLabV3 
and FCN models.

**What stays the same:** same as above.

---

### Generative models (e.g. diffusion models, VAEs)
There is no class label — the target is the image itself (or a denoised version of it). 
The loss function changes (MSE for reconstruction, or a more complex denoising 
objective for diffusion). The validation metric changes — FID score or visual 
inspection rather than accuracy.

**What stays the same:** data loading, training loop structure, checkpointing.

---

### Self-supervised learning (e.g. SimCLR, CLIP)
The DataLoader needs to return pairs or multiple views of each image rather than 
image-label pairs. The loss function is contrastive (e.g. InfoNCE). There is no 
validation accuracy in the usual sense — evaluation is done by training a linear 
classifier on frozen features.

**What stays the same:** model definition pattern, training loop structure, 
checkpointing.

---

**The key insight:** the five-step training loop (`zero_grad` → forward → loss → 
`backward` → `step`) never changes. What changes between tasks is the model 
architecture, the loss function, and what the DataLoader returns. Everything else 
in this notebook can be reused almost verbatim.

---
## Your task for today

You have spent the first part of this session reading through and running this 
pipeline on CIFAR-10. Now spend the rest of the session adapting it to your own project.

Work through these steps in order:

1. **Get your data into the pipeline.** Use `ImageFolder` if your data is already 
   organised by class, or adapt `CustomImageDataset` if not. Run `sanity_check_dataset` 
   and make sure the images and labels look correct.

2. **Choose your model.** ResNet-18 is a good default. If your task is not classification, 
   see section 9 for pointers. Ask a TA if you are unsure.

3. **Run a quick training experiment.** Set `NUM_EPOCHS = 3` to start. The goal is 
   not a good result — just to confirm that the pipeline runs without errors and the 
   loss decreases.

4. **Look at your loss curves.** Do they look healthy? If something looks wrong, 
   ask a TA.

By the end of today's session, every group should have a pipeline that runs on 
their own data. This is your foundation for the rest of the semester.